# Strands Agents with Amazon Neptune MCP Server Demo

This notebook demonstrates how to use [Strands Agents](https://strandsagents.com/) with the [Amazon Neptune MCP Server](https://awslabs.github.io/mcp/servers/amazon-neptune-mcp-server/) to intelligently interact with Amazon Neptune graphs.

## Overview

This demo showcases:
1. **Neptune MCP Server Setup**: Connect to Neptune Analytics or Neptune Database
2. **Graph Status Monitoring**: Check if the graph is available
3. **Schema Discovery**: Retrieve and analyze graph structure
4. **Intelligent Querying**: Use Strands Agents to generate and execute openCypher queries
5. **Autonomous Graph Analysis**: Let the agent explore and analyze graph data

## Architecture

```
User Prompt → Strands Agent → Neptune MCP Server → Neptune Graph
                    ↓
              Amazon Bedrock
              (Claude Sonnet 5.5)
```

## 1. Setup

Install the dependencies. `uv` provides the `uvx` command used to run the Neptune MCP server.

Prerequisites:
- AWS credentials with access to your Neptune graph and to Amazon Bedrock
- Access to the Claude Sonnet model in Amazon Bedrock
- Network access from this notebook to Neptune (Neptune Database runs in a VPC; Neptune Analytics can use a public endpoint)

In [ ]:
%pip install -q strands-agents uv

In [ ]:
import os
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters
import boto3

## 2. Configuration

In [ ]:
# Configuration
# For Neptune Analytics, use format: neptune-graph://<graph-identifier>
# For Neptune Database, use format: neptune-db://<cluster-endpoint>

NEPTUNE_ENDPOINT = "neptune-graph://<INPUT_GRAPH_ID>"
AWS_REGION = "us-west-2"
AWS_PROFILE = None

# Bedrock configuration
# Claude Sonnet 5.5 through the US cross-region inference profile
BEDROCK_MODEL = "us.anthropic.claude-sonnet-5-5"
BEDROCK_REGION = "us-west-2"

# Set environment variables for Neptune MCP server
os.environ['NEPTUNE_ENDPOINT'] = NEPTUNE_ENDPOINT
os.environ['AWS_REGION'] = AWS_REGION
if AWS_PROFILE:
    os.environ['AWS_PROFILE'] = AWS_PROFILE

print(f"Configuration set:")
print(f"  Neptune Endpoint: {NEPTUNE_ENDPOINT}")
print(f"  AWS Region: {AWS_REGION}")
print(f"  Bedrock Model: {BEDROCK_MODEL}")

## 3. Initialize Neptune MCP Client

The Neptune MCP server provides four tools:
- `get_graph_status`: Check if the graph is available
- `get_graph_schema`: Retrieve the graph schema
- `run_opencypher_query`: Execute an openCypher query
- `run_gremlin_query`: Execute a Gremlin query (Neptune Database only)

In [ ]:
# Initialize Neptune MCP Client
def create_neptune_mcp_client():
    """Create and return a Neptune MCP client."""
    env = {
        "NEPTUNE_ENDPOINT": NEPTUNE_ENDPOINT,
        "AWS_REGION": AWS_REGION,
        "FASTMCP_LOG_LEVEL": "INFO",
    }
    # The MCP server process only inherits a few basic variables (HOME, PATH, ...),
    # so pass along the AWS profile or credentials explicitly when they are set.
    for var in ("AWS_PROFILE", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"):
        if os.environ.get(var):
            env[var] = os.environ[var]

    return MCPClient(lambda: stdio_client(
        StdioServerParameters(
            command="uvx",
            args=["awslabs.amazon-neptune-mcp-server@latest"],
            env=env,
        )
    ))

print("Neptune MCP client creator ready")

## 4. Direct Neptune MCP Operations

Let's test basic Neptune operations using the MCP server directly.
reference: https://strandsagents.com/latest/documentation/docs/api-reference/tools/

In [ ]:
# Test Neptune MCP server directly
print("Testing Neptune MCP Server...\n")

with create_neptune_mcp_client() as neptune_mcp:
    # List available tools
    tools = neptune_mcp.list_tools_sync()
    print(f"Available Neptune MCP Tools: {len(tools)}")

    for tool in tools:
        print(f"  - {tool.tool_name}: {tool.tool_spec}")
    print()

## 5. Create Strands Agent with Neptune MCP Tools

Now let's create an intelligent agent that can interact with Neptune autonomously.

In [ ]:

NEPTUNE_AGENT_SYSTEM_PROMPT = """
You are an expert Neptune graph database analyst and assistant.

Your capabilities include:
1. Checking the status and availability of Neptune graphs
2. Analyzing graph schemas to understand data structure
3. Generating and executing openCypher queries to retrieve information
4. Providing insights and analysis about graph data

When analyzing graphs:
- Always start by checking the graph status and schema
- Generate efficient openCypher queries based on the schema
- Explain your findings clearly with relevant statistics
- Provide actionable insights from the data

When writing openCypher queries:
- Use proper syntax for Neptune Analytics/Database
- Include LIMIT clauses to avoid overwhelming results
- Handle potential errors gracefully
- Explain what each query does before executing it
"""

print("Agent system prompt defined")

In [ ]:
# Create the Neptune Graph Analysis Agent
bedrock_model = BedrockModel(
    model_id=BEDROCK_MODEL,
    boto_session=boto3.Session(profile_name=AWS_PROFILE, region_name=BEDROCK_REGION),
)

def create_neptune_agent(neptune_mcp):
    """Create a Strands Agent that uses the tools of a started Neptune MCP client.

    Call this inside a `with create_neptune_mcp_client() as neptune_mcp:` block,
    and use the agent before the block ends, while the MCP session is open.
    """
    return Agent(
        model=bedrock_model,
        system_prompt=NEPTUNE_AGENT_SYSTEM_PROMPT,
        tools=neptune_mcp.list_tools_sync(),
    )

print("Neptune agent creator ready")

## 6. Agent Demonstrations

Let's see the agent in action with various tasks.

### 6.1 Basic Graph Health Check

In [ ]:
print("=" * 80)
print("TASK 1: Graph Health Check")
print("=" * 80)

with create_neptune_mcp_client() as neptune_mcp:
    agent = create_neptune_agent(neptune_mcp)

    prompt = """
    Check the health and status of the Neptune graph. 
    Provide a summary of its availability and readiness.
    """
    
    print(f"\nUser Prompt: {prompt.strip()}\n")
    print("Agent Response:")
    print("-" * 80)
    
    response = agent(prompt)
    print(response)
    print("\n" + "=" * 80 + "\n")

### 6.2 Schema Analysis and Exploration

In [ ]:
print("=" * 80)
print("TASK 2: Schema Analysis")
print("=" * 80)

with create_neptune_mcp_client() as neptune_mcp:
    agent = create_neptune_agent(neptune_mcp)

    prompt = """
    Analyze the Neptune graph schema and provide:
    1. A summary of the node types (labels) present
    2. A summary of the relationship types present
    3. Key properties for important node and relationship types
    4. An overview of the graph structure and domain it represents
    """
    
    print(f"\nUser Prompt: {prompt.strip()}\n")
    print("Agent Response:")
    print("-" * 80)
    
    response = agent(prompt)
    print(response)
    print("\n" + "=" * 80 + "\n")

### 6.3 Data Exploration and Statistics

In [ ]:
print("=" * 80)
print("TASK 3: Data Exploration and Statistics")
print("=" * 80)

with create_neptune_mcp_client() as neptune_mcp:
    agent = create_neptune_agent(neptune_mcp)

    prompt = """
    Explore the Neptune graph and provide statistics:
    1. Total count of nodes and relationships
    2. Count of nodes by type/label
    3. Count of relationships by type
    4. Any interesting patterns or insights you notice
    
    Please execute appropriate openCypher queries to gather this information.
    """
    
    print(f"\nUser Prompt: {prompt.strip()}\n")
    print("Agent Response:")
    print("-" * 80)
    
    response = agent(prompt)
    print(response)
    print("\n" + "=" * 80 + "\n")

### 6.4 Natural Language Query Execution

In [ ]:
print("=" * 80)
print("TASK 4: Natural Language Query")
print("=" * 80)

with create_neptune_mcp_client() as neptune_mcp:
    agent = create_neptune_agent(neptune_mcp)

    prompt = """
    Find the most connected nodes in the graph. 
    Show me the top 5 nodes with the most relationships,
    including their labels and the count of connections.
    """
    
    print(f"\nUser Prompt: {prompt.strip()}\n")
    print("Agent Response:")
    print("-" * 80)
    
    response = agent(prompt)
    print(response)
    print("\n" + "=" * 80 + "\n")

### 6.5 Complex Graph Analysis

In [ ]:
# Task 5: Complex analysis
print("=" * 80)
print("TASK 5: Complex Graph Analysis")
print("=" * 80)

with create_neptune_mcp_client() as neptune_mcp:
    agent = create_neptune_agent(neptune_mcp)

    # This prompt should be customized based on your graph domain
    prompt = """
    Perform a comprehensive analysis of the graph:
    1. Identify the most important node types based on connectivity
    2. Find any obvious patterns in the relationship structure
    3. Look for potential data quality issues (disconnected nodes, missing properties, etc.)
    4. Provide recommendations for graph queries that would be useful for this data
    
    Be thorough and execute multiple queries as needed to gather insights.
    """
    
    print(f"\nUser Prompt: {prompt.strip()}\n")
    print("Agent Response:")
    print("-" * 80)
    
    response = agent(prompt)
    print(response)
    print("\n" + "=" * 80 + "\n")